# Dissatisfaction-Risk Feature Engineering

This notebook prepares the modeling dataset for predicting customer dissatisfaction risk.

The original escalation-risk approach was revised because `is_escalated` could not be defensibly derived from the available dataset. The predictive target is therefore `is_dissatisfied`, created from observed Customer Satisfaction Rating values:

- `1` = Customer Satisfaction Rating ≤ 2
- `0` = Customer Satisfaction Rating ≥ 3
- Missing = Customer Satisfaction Rating unavailable

The model will be trained only on tickets with an observed CSAT value and later applied to the full ticket population.

Only features selected as suitable for dataset-wide dissatisfaction-risk inference are used for modeling. Fields that directly define the target, reveal CSAT availability, represent post-resolution outcomes, or otherwise create a substantial risk of leakage are excluded.

## Modeling Scope

### Training population

Tickets where:

`Has CSAT = 1`

These tickets have an observed `is_dissatisfied` target and can be used for supervised model training and evaluation.

### Inference population

All tickets in the cleaned dataset.

The final model will generate dissatisfaction-risk scores for tickets regardless of whether an observed CSAT value is available.

### Candidate feature groups

**Ticket metadata**
- Ticket Priority
- Ticket Channel
- First-response availability

**AI-derived features**
- Sentiment label
- Sentiment score
- Topic assignment

### Excluded fields

The following fields are excluded from the predictive feature set because they are post-resolution outcomes, directly related to CSAT, or otherwise unsuitable for inference:

- Customer Satisfaction Rating
- Has CSAT
- is_dissatisfied
- Resolution
- Time to Resolution

## 1. Imports and Configuration

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

## 2. Load and Validate Source Data

In [2]:
# Define the project root relative to the notebooks directory.
PROJECT_ROOT = Path.cwd().parent

# Define paths to the cleaned ticket data and AI-derived outputs.
CLEANED_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "customer_support_tickets_cleaned.csv"
)

SENTIMENT_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "ticket_sentiment.csv"
)

TOPIC_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "customer_support_topics.csv"
)

# Load the source datasets.
cleaned_df = pd.read_csv(CLEANED_DATA_PATH)
sentiment_df = pd.read_csv(SENTIMENT_DATA_PATH)
topic_df = pd.read_csv(TOPIC_DATA_PATH)

# Confirm the expected dataset sizes before merging.
print("Cleaned data shape:", cleaned_df.shape)
print("Sentiment data shape:", sentiment_df.shape)
print("Topic data shape:", topic_df.shape)

Cleaned data shape: (8469, 20)
Sentiment data shape: (8469, 3)
Topic data shape: (8469, 3)


In [3]:
# Preview the cleaned ticket data.
display(cleaned_df.head())

# Preview the sentiment predictions.
display(sentiment_df.head())

# Preview the topic assignments.
display(topic_df.head())

,Ticket ID,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating,Cleaned Ticket Description,Has First Response,Is Resolved,Has CSAT,is_dissatisfied
0,1,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN,I'm having an issue with the GoPro Hero. Pleas...,1,0,0,NaN
1,2,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN,I'm having an issue with the LG Smart TV. Plea...,1,0,0,NaN
2,3,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0,I'm facing a problem with my Dell XPS. The Del...,1,1,1,0.0
3,4,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0,I'm having an issue with the Microsoft Office....,1,1,1,0.0
4,5,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0,I'm having an issue with the Autodesk AutoCAD....,1,1,1,1.0


,ticket_id,sentiment_label,sentiment_score
0,1,NEGATIVE,0.994507
1,2,NEGATIVE,0.948731
2,3,NEGATIVE,0.999365
3,4,NEGATIVE,0.967981
4,5,NEGATIVE,0.997396


,Ticket ID,topic_id,topic_label
0,1,20,Repair or Replacement Concerns
1,2,0,Issue After Firmware Update
2,3,-1,Other or Unclassified Support Content
3,4,0,Issue After Firmware Update
4,5,11,Software Bugs and Data Loss


## 3. Validate Ticket-Level Join Keys

In [4]:
# Display the column names for each source dataset.
print("Cleaned data columns:")
print(cleaned_df.columns.tolist())

print("\nSentiment data columns:")
print(sentiment_df.columns.tolist())

print("\nTopic data columns:")
print(topic_df.columns.tolist())

Cleaned data columns:
['Ticket ID', 'Customer Age', 'Customer Gender', 'Product Purchased', 'Date of Purchase', 'Ticket Type', 'Ticket Subject', 'Ticket Description', 'Ticket Status', 'Resolution', 'Ticket Priority', 'Ticket Channel', 'First Response Time', 'Time to Resolution', 'Customer Satisfaction Rating', 'Cleaned Ticket Description', 'Has First Response', 'Is Resolved', 'Has CSAT', 'is_dissatisfied']

Sentiment data columns:
['ticket_id', 'sentiment_label', 'sentiment_score']

Topic data columns:
['Ticket ID', 'topic_id', 'topic_label']


In [5]:
# Standardize the sentiment dataset join key to match the other source datasets.
sentiment_df = sentiment_df.rename(
    columns={"ticket_id": "Ticket ID"}
)

# Confirm the standardized column names.
print("Sentiment data columns:")
print(sentiment_df.columns.tolist())

Sentiment data columns:
['Ticket ID', 'sentiment_label', 'sentiment_score']


In [6]:
# Store the source datasets for consistent ticket-level validation.
source_datasets = {
    "Cleaned": cleaned_df,
    "Sentiment": sentiment_df,
    "Topic": topic_df,
}

# Check row counts, unique ticket IDs, missing join keys, and duplicates.
for name, dataframe in source_datasets.items():
    print(f"{name} dataset")
    print("Rows:", len(dataframe))
    print("Unique Ticket IDs:", dataframe["Ticket ID"].nunique())
    print("Missing Ticket IDs:", dataframe["Ticket ID"].isna().sum())
    print("Duplicate Ticket IDs:", dataframe["Ticket ID"].duplicated().sum())
    print("-" * 40)

Cleaned dataset
Rows: 8469
Unique Ticket IDs: 8469
Missing Ticket IDs: 0
Duplicate Ticket IDs: 0
----------------------------------------
Sentiment dataset
Rows: 8469
Unique Ticket IDs: 8469
Missing Ticket IDs: 0
Duplicate Ticket IDs: 0
----------------------------------------
Topic dataset
Rows: 8469
Unique Ticket IDs: 8469
Missing Ticket IDs: 0
Duplicate Ticket IDs: 0
----------------------------------------


In [7]:
# Convert ticket IDs to sets to verify that all source datasets contain
# the same ticket population before merging.
cleaned_ids = set(cleaned_df["Ticket ID"])
sentiment_ids = set(sentiment_df["Ticket ID"])
topic_ids = set(topic_df["Ticket ID"])

print(
    "Cleaned and sentiment ticket IDs match:",
    cleaned_ids == sentiment_ids
)

print(
    "Cleaned and topic ticket IDs match:",
    cleaned_ids == topic_ids
)

Cleaned and sentiment ticket IDs match: True
Cleaned and topic ticket IDs match: True


## 4. Assemble the Modeling Dataset

In [8]:
# Merge sentiment predictions with the cleaned ticket dataset.
modeling_df = cleaned_df.merge(
    sentiment_df,
    on="Ticket ID",
    how="left",
    validate="one_to_one",
)

# Merge topic assignments with the combined dataset.
modeling_df = modeling_df.merge(
    topic_df,
    on="Ticket ID",
    how="left",
    validate="one_to_one",
)

# Confirm the shape of the combined ticket-level dataset.
print("Combined dataset shape:", modeling_df.shape)

Combined dataset shape: (8469, 24)


In [9]:
# Verify that the combined dataset preserved the full ticket population
# and that all AI-derived features are available after merging.
print("Rows:", len(modeling_df))
print("Unique Ticket IDs:", modeling_df["Ticket ID"].nunique())
print("Duplicate Ticket IDs:", modeling_df["Ticket ID"].duplicated().sum())

print("\nMissing AI-derived values:")
print(
    modeling_df[
        [
            "sentiment_label",
            "sentiment_score",
            "topic_id",
            "topic_label",
        ]
    ].isna().sum()
)

Rows: 8469
Unique Ticket IDs: 8469
Duplicate Ticket IDs: 0

Missing AI-derived values:
sentiment_label    0
sentiment_score    0
topic_id           0
topic_label        0
dtype: int64


## 5. Define Training and Inference Populations

In [10]:
# Retain the full ticket population for future dissatisfaction-risk inference.
inference_df = modeling_df.copy()

# Restrict supervised model development to tickets with an observed CSAT value.
training_df = modeling_df[
    modeling_df["Has CSAT"] == 1
].copy()

# Confirm the size of the training and inference populations.
print("Training population:", training_df.shape)
print("Inference population:", inference_df.shape)

# Verify that the target is fully observed in the training population.
print("\nMissing target values in training population:")
print(training_df["is_dissatisfied"].isna().sum())

Training population: (2769, 24)
Inference population: (8469, 24)

Missing target values in training population:
0


In [11]:
# Compare CSAT availability across the training and full inference populations.
print("CSAT availability in full dataset:")
print(
    inference_df["Has CSAT"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nTarget distribution in training population:")
print(
    training_df["is_dissatisfied"]
    .value_counts(dropna=False)
    .sort_index()
)

CSAT availability in full dataset:
Has CSAT
0    5700
1    2769
Name: count, dtype: int64

Target distribution in training population:
is_dissatisfied
0.0    1667
1.0    1102
Name: count, dtype: int64


## 6. Define Modeling Features and Exclude Leakage

The predictive feature set is restricted to information that can be available before or during ticket handling without using the observed CSAT outcome or post-resolution results.

The target is `is_dissatisfied`.

Customer Satisfaction Rating and Has CSAT are excluded because they directly define or reveal target availability. Resolution and Time to Resolution are excluded because they represent post-resolution information.

Ticket identifiers and free-text fields are also excluded from this structured feature set. Ticket content has already been represented through the sentiment and topic features generated in the previous analysis steps.

In [12]:
# Document columns intentionally excluded to prevent target leakage
# or reliance on post-resolution information.
excluded_columns = [
    "Ticket ID",
    "Customer Satisfaction Rating",
    "Has CSAT",
    "is_dissatisfied",
    "Resolution",
    "Time to Resolution",
    "Ticket Description",
    "Cleaned Ticket Description",
]

# Define the initial candidate feature set using ticket metadata
# and AI-derived sentiment and topic outputs.
candidate_features = [
    "Ticket Priority",
    "Ticket Channel",
    "Has First Response",
    "sentiment_label",
    "sentiment_score",
    "topic_id",
]

# Confirm that all selected features are available in the training dataset.
missing_features = [
    feature
    for feature in candidate_features
    if feature not in training_df.columns
]

print("Candidate features:")
print(candidate_features)

print("\nMissing candidate features:")
print(missing_features)

Candidate features:
['Ticket Priority', 'Ticket Channel', 'Has First Response', 'sentiment_label', 'sentiment_score', 'topic_id']

Missing candidate features:
[]


## 7. Inspect Candidate Feature Quality

The selected features are reviewed for missing values and basic distributions before preprocessing.

This check helps identify whether imputation or additional handling is required before creating the modeling pipeline.

In [13]:
# Create a feature-level missing-value summary for the training population.
feature_missingness = (
    training_df[candidate_features]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("Missing values by candidate feature:")
print(feature_missingness)

Missing values by candidate feature:
Ticket Priority       0
Ticket Channel        0
Has First Response    0
sentiment_label       0
sentiment_score       0
topic_id              0
dtype: int64


In [14]:
# Review categorical feature distributions in the training population.
categorical_features = [
    "Ticket Priority",
    "Ticket Channel",
    "sentiment_label",
]

for feature in categorical_features:
    print(f"\n{feature}")
    print(training_df[feature].value_counts(dropna=False))


Ticket Priority
Ticket Priority
Critical    726
High        705
Medium      694
Low         644
Name: count, dtype: int64

Ticket Channel
Ticket Channel
Email           720
Phone           691
Social media    684
Chat            674
Name: count, dtype: int64

sentiment_label
sentiment_label
NEGATIVE    2521
POSITIVE     248
Name: count, dtype: int64


In [15]:
# Summarize the indicator and continuous numeric candidate features.
numeric_features = [
    "Has First Response",
    "sentiment_score",
]

print(
    training_df[numeric_features]
    .describe()
)

# Review topic assignment counts separately because topic IDs are categorical labels.
print("\ntopic_id")
print(training_df["topic_id"].value_counts(dropna=False).sort_index())

       Has First Response  sentiment_score
count              2769.0      2769.000000
mean                  1.0         0.975047
std                   0.0         0.075321
min                   1.0         0.501304
25%                   1.0         0.992409
50%                   1.0         0.998180
75%                   1.0         0.999440
max                   1.0         0.999818

topic_id
topic_id
-1     688
 0     596
 1     127
 2     119
 3     120
 4     123
 5      95
 6      86
 7     106
 8      64
 9      60
 10     59
 11     71
 12     50
 13     43
 14     55
 15     44
 16     46
 17     40
 18     39
 19     32
 20     32
 21     27
 22     20
 23      4
 24      5
 25      7
 26      5
 27      3
 28      3
Name: count, dtype: int64


## 8. Finalize the Modeling Feature Set

Feature-quality checks showed that `Has First Response` has no variation within the CSAT-observed training population. Because a constant feature cannot contribute to prediction, it is excluded from the final modeling feature set.

`topic_id` is treated as a categorical feature because BERTopic topic identifiers are model-generated labels and do not represent an ordinal scale.

The final feature set combines ticket metadata with AI-derived sentiment and topic features.

In [16]:
# Define the final categorical and numeric feature groups.

categorical_features = [
    "Ticket Priority",
    "Ticket Channel",
    "sentiment_label",
    "topic_id",
]

numeric_features = [
    "sentiment_score",
]

# Combine the feature groups into the final base feature set.
model_features = categorical_features + numeric_features

print("Final modeling features:")
print(model_features)

print("\nNumber of base features:", len(model_features))

Final modeling features:
['Ticket Priority', 'Ticket Channel', 'sentiment_label', 'topic_id', 'sentiment_score']

Number of base features: 5


## 9. Evaluate a Priority and Sentiment Interaction

A potential interaction feature is evaluated by comparing dissatisfaction rates across combinations of ticket priority and sentiment.

The interaction will only be added if it provides a clear and interpretable distinction beyond the individual features.

In [17]:
# Calculate dissatisfaction rates across priority and sentiment combinations.
priority_sentiment_rates = (
    training_df
    .groupby(
        ["Ticket Priority", "sentiment_label"],
        observed=True
    )["is_dissatisfied"]
    .agg(["count", "mean"])
    .rename(columns={"mean": "dissatisfaction_rate"})
    .sort_values("dissatisfaction_rate", ascending=False)
)

print(priority_sentiment_rates)

                                 count  dissatisfaction_rate
Ticket Priority sentiment_label                             
Critical        POSITIVE            57              0.438596
High            POSITIVE            62              0.419355
                NEGATIVE           643              0.409020
Critical        NEGATIVE           669              0.408072
Medium          NEGATIVE           627              0.395534
Low             NEGATIVE           582              0.384880
                POSITIVE            62              0.354839
Medium          POSITIVE            67              0.313433


## 10. Examine Target Class Balance

Class balance is evaluated using the CSAT-observed training population.

The dissatisfied class is expected to be moderately smaller than the non-dissatisfied class, so the train/test split will preserve the target distribution through stratification.

A DummyClassifier baseline will provide a reference performance level before training predictive models.

In [18]:
# Calculate target counts and proportions in the training population.
class_balance = (
    training_df["is_dissatisfied"]
    .value_counts()
    .sort_index()
    .rename_axis("is_dissatisfied")
    .reset_index(name="count")
)

# Add the percentage represented by each target class.
class_balance["percentage"] = (
    class_balance["count"] / len(training_df) * 100
)

print(class_balance)

   is_dissatisfied  count  percentage
0              0.0   1667   60.202239
1              1.0   1102   39.797761


In [19]:
# Calculate the majority-class baseline for reference.
majority_class_percentage = (
    training_df["is_dissatisfied"]
    .value_counts(normalize=True)
    .max()
    * 100
)

print(
    f"\nMajority-class proportion: "
    f"{majority_class_percentage:.2f}%"
)


Majority-class proportion: 60.20%


## 11. Create Stratified Training and Test Sets

The CSAT-observed population is split into training and test sets using stratification to preserve the dissatisfaction-class distribution.

The test set will remain untouched during baseline preparation and will be used for model evaluation in the next stage.

In [20]:
# Separate the selected predictive features from the dissatisfaction target.
X = training_df[model_features].copy()
y = training_df["is_dissatisfied"].astype(int)

# Create a stratified train/test split to preserve the target distribution.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

# Confirm the resulting dataset sizes.
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\ny_train distribution:")
print(y_train.value_counts(normalize=True).sort_index())

print("\ny_test distribution:")
print(y_test.value_counts(normalize=True).sort_index())

X_train shape: (2215, 5)
X_test shape: (554, 5)

y_train distribution:
is_dissatisfied
0    0.601806
1    0.398194
Name: proportion, dtype: float64

y_test distribution:
is_dissatisfied
0    0.602888
1    0.397112
Name: proportion, dtype: float64


## 12. Build the Feature Preprocessing Pipeline

Categorical features are transformed using one-hot encoding so they can be used by classification models.

`sentiment_score` is retained as a numeric feature.

The preprocessing steps are placed in a reusable pipeline so the same transformations can later be applied consistently to both training and inference data.

In [21]:
# Define the categorical and numeric feature groups used for preprocessing.
categorical_features = [
    "Ticket Priority",
    "Ticket Channel",
    "sentiment_label",
    "topic_id",
]

numeric_features = [
    "sentiment_score",
]

# One-hot encode categorical features and pass numeric features through unchanged.
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features,
        ),
        (
            "numeric",
            "passthrough",
            numeric_features,
        ),
    ]
)

In [22]:
# Fit the preprocessing steps on the training features only.
X_train_transformed = preprocessor.fit_transform(X_train)

# Apply the fitted preprocessing steps to the untouched test features.
X_test_transformed = preprocessor.transform(X_test)

# Confirm the transformed feature matrix sizes.
print("Transformed X_train shape:", X_train_transformed.shape)
print("Transformed X_test shape:", X_test_transformed.shape)

Transformed X_train shape: (2215, 41)
Transformed X_test shape: (554, 41)


## 13. Establish a Baseline Model

A DummyClassifier is used to establish a simple reference point before training predictive models.

The baseline uses the most frequent target class as its prediction strategy. Future classification models will be evaluated against this baseline using the same stratified training and test split.

In [23]:
# Initialize a baseline classifier that always predicts the most frequent class.
dummy_classifier = DummyClassifier(
    strategy="most_frequent",
    random_state=42,
)

# Fit the baseline classifier on the transformed training data.
dummy_classifier.fit(
    X_train_transformed,
    y_train,
)

# Generate baseline predictions for the test set.
dummy_predictions = dummy_classifier.predict(
    X_test_transformed
)

# Display the baseline classification metrics.
print(
    classification_report(
        y_test,
        dummy_predictions,
        zero_division=0,
    )
)

              precision    recall  f1-score   support

           0       0.60      1.00      0.75       334
           1       0.00      0.00      0.00       220

    accuracy                           0.60       554
   macro avg       0.30      0.50      0.38       554
weighted avg       0.36      0.60      0.45       554



In [24]:
# Calculate and display the baseline accuracy separately for reference.
dummy_accuracy = dummy_classifier.score(
    X_test_transformed,
    y_test,
)

print(f"Dummy baseline accuracy: {dummy_accuracy:.4f}")

Dummy baseline accuracy: 0.6029


## 14. Modeling Dataset Summary

The final modeling dataset combines cleaned ticket metadata with AI-derived sentiment and topic features.

### Training population

The supervised modeling population contains only tickets with an observed Customer Satisfaction Rating.

- Total tickets: 2,769
- Not dissatisfied: 1,667 (60.20%)
- Dissatisfied: 1,102 (39.80%)

### Inference population

The full dataset contains 8,469 tickets.

The final model will later be applied to all tickets to generate dissatisfaction-risk predictions, including tickets without an observed CSAT value.

### Final feature set

Categorical features:

- Ticket Priority
- Ticket Channel
- sentiment_label
- topic_id

Numeric feature:

- sentiment_score

`Has First Response` was excluded because it had no variation within the CSAT-observed training population.

No custom interaction feature was added because the priority and sentiment combination analysis did not show a sufficiently strong or consistent pattern to justify additional feature complexity.

### Leakage exclusions

The predictive feature set excludes:

- Ticket ID
- Customer Satisfaction Rating
- Has CSAT
- is_dissatisfied
- Resolution
- Time to Resolution
- Ticket Description
- Cleaned Ticket Description

These exclusions prevent the model from directly using the target, target-related fields, post-resolution outcomes, or raw text already represented through sentiment and topic features.

### Train/test split

The CSAT-observed population was split using an 80/20 stratified split.

- Training set: 2,215 tickets
- Test set: 554 tickets

The dissatisfaction-class proportions were preserved across both sets.

### Baseline performance

A DummyClassifier using the most-frequent-class strategy achieved:

- Accuracy: 60.29%
- Dissatisfied-class recall: 0.00
- Dissatisfied-class F1-score: 0.00

This baseline demonstrates that accuracy alone is not sufficient for evaluating the dissatisfaction-risk model. Future models will be compared using precision, recall, and F1-score, with particular attention to identifying dissatisfied tickets.

## 15. Final Validation

In [25]:
# Validate the final feature configuration and modeling dataset sizes.
print("Training population shape:", training_df.shape)
print("Inference population shape:", inference_df.shape)

print("\nModeling features:")
for feature in model_features:
    print(f"- {feature}")

print("\nTrain/test split:")
print("Training features:", X_train.shape)
print("Test features:", X_test.shape)

print("\nTransformed feature matrices:")
print("Training matrix:", X_train_transformed.shape)
print("Test matrix:", X_test_transformed.shape)

print(f"\nDummy baseline accuracy: {dummy_accuracy:.4f}")

Training population shape: (2769, 24)
Inference population shape: (8469, 24)

Modeling features:
- Ticket Priority
- Ticket Channel
- sentiment_label
- topic_id
- sentiment_score

Train/test split:
Training features: (2215, 5)
Test features: (554, 5)

Transformed feature matrices:
Training matrix: (2215, 41)
Test matrix: (554, 41)

Dummy baseline accuracy: 0.6029
